# GroupNorm

The PyTorch [documentation for GroupNorm](https://docs.pytorch.org/docs/main/generated/torch.nn.modules.normalization.GroupNorm.html) says that it computes the following:

$$
y = \frac{x - \text{E}[x]}{\sqrt{\text{Var}[x] + \epsilon}} \gamma + \beta
$$

Where $x$ and $y$ are both shape $(N, C, *)$, where $C$ is the number of channels. The $\text{E}[x]$ and $\text{Var}[x]$ reductions are calculated over the $C$ dimension and any dimensions following it. But crucially they are not reductions over the entire channel dimension. Instead, the channel dimension is divided into `num_groups` different reduction groups ($C$ must be divisible by `num_groups`) over which separate reductions are computed.

So for instance, if the input is shape $(1, 4, 3, 5)$ and `num_groups=2`, then we have two different reductions, one over the elements `x[0, :2, :, :]` and the other over the elements `x[0, 2:, :, :]`.

For one group indexed by $i$, with $m$ group elements labeled $x_k$, the reduction operations are just the mean and variance of the group elements:

$$
\text{E}[x]_i = \frac{1}{m} \sum x_k
$$

$$
\text{Var}[x]_i = \frac{1}{m} \sum (x_k - \text{E}[x]_i)^2
$$

$\gamma$ (`weight`) and $\beta$ (`bias`) are both optional. If given, they both have size $(C,)$.

$\epsilon$ (`eps`) is also optional, and it is a float.

In [68]:
import torch
import math

def my_group_norm(x, num_groups, eps):
    N = x.size(0)
    C = x.size(1)
    assert (C % num_groups) == 0
    C_per_group = int(C / num_groups)
    # Split the channel dimension (C,) --> (num_groups, C_per_group)
    x_view = x.view([N, num_groups, C_per_group, *x.shape[2:]])
    elem_per_group = x.numel() / (N * num_groups)
    # Perform reductions over the groups
    reduction_dims = list(range(2, x_view.dim()))
    mean = x_view.mean(dim=reduction_dims, keepdim=True)
    var = torch.sum((x_view - mean)**2, dim=reduction_dims, keepdim=True) / elem_per_group
    y_view = (x_view - mean) / torch.sqrt(var + eps)
    # Recombine the channel dim (num_groups, C_per_group) --> (C,)
    y = y_view.flatten(1, 2)
    return y


In [97]:
import itertools

cases = [
    # (shape, num_groups)
    ((1, 2), 2),
    ((1, 2, 4, 8), 2),
    ((2, 2, 4, 8), 2),
    ((2, 4, 4, 8), 2),
    ((2, 8, 4, 8), 2),
    ((10, 90, 40, 8, 15), 3),
]

eps_list = [
    1e-05,
    1e-02,
    100,
]

for (shape, num_groups), eps in itertools.product(cases, eps_list):
    x = torch.randn(shape)
    y = my_group_norm(x, num_groups, eps=eps)
    y_check = torch.group_norm(x, num_groups, eps=eps)

    assert y.shape == y_check.shape
    assert torch.allclose(y, y_check, rtol=1e-4, atol=1e-4)